## 📑 HAPS L3869 Modifications
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/BYDGHQ)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()


from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os

#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ"S3PREFIX" 

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `p-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

###### Schema of the table

In [0]:
# Define the schema name
schema_name = f"`{env_prefix}-cdh-delivery`.repoint"

# Create the schema if it does not exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

table_name = 'HAPS_L3869Modifications'

schema = (
"BusinessDate date," 
"PortfolioName string,"
"Opendate date,"
"AssetSegment string," 
"CDI string,"
"PartyTaxRegistrationNumber string,"
"ExposureKey string,"
"AgreementKey string,"
"ProductKey string,"
"TotalLegalClaimEUR decimal(19,2),"
"GBVEUR decimal(19,2),"
"OnBalanceAmountEUR decimal(19,2),"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""

# ================== Create Delta Table ==================
#delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"  
#AWS Credentials not available thus env_prefix returns empty temporarily we manually set it to `qa`
#env_prefix = 'qa'
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date
#### type: ignore
)


##### Create Delta Tables with selected fields & Temporary Views for each delta table

##### haps_portfolios

In [0]:
haps_portfolios = spark.sql(
f'''
SELECT DISTINCT A.PortfolioName
FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios  A
INNER JOIN (SELECT BusinessDate, MAX(IngestionTimeStamp) IngestionTimeStamp, PortfolioName 
            FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios
            GROUP BY  BusinessDate, PortfolioName) B
ON A.BusinessDate = B.BusinessDate AND A.PortfolioName = B.PortfolioName AND A.IngestionTimeStamp = B.IngestionTimeStamp
WHERE A.Report = 'HAPS_L3869Modifications'
AND A.IsActive = 1 
''' 
)
haps_portfolios.createOrReplaceTempView("haps_portfolios")

###### dexposure

In [0]:

dexposure = spark.sql(
f'''
SELECT 
    A.BusinessDate
    , A.PortfolioName
    , A.ExposureKey
    , A.Opendate
    , A.SourceSystem
    , A.AssetSegment
    , A.CDI
    , A.PartyTaxRegistrationNumber
    , A.AgreementKey
    , A.ProductKey
    , A.OutstandingTotalBalance
    , A.GBV
    , A.OnBalance
    , A.Currency
    , A.IsActiveFlag
FROM `{env_prefix}-cdh`.derivative.dexposure as A
INNER JOIN haps_portfolios AS HP
ON HP.PortfolioName = A.PortfolioName
WHERE A.BusinessDate = {run_date!r}
AND A.IsActiveFlag = true
'''
)
dexposure.createOrReplaceTempView("dexposure")

###### modifications

In [0]:
modifications = spark.sql(
f'''
SELECT 
  A.BusinessDate
, A.PortfolioName
, A.Opendate 
, A.AssetSegment
, A.CDI
, A.PartyTaxRegistrationNumber
, A.ExposureKey
, A.AgreementKey
, A.ProductKey
, `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.OutstandingTotalBalance, A.BusinessDate) as `TotalLegalClaimEUR`
, `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.GBV, A.BusinessDate) as `GBVEUR`
, `{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.OnBalance, A.BusinessDate) as `OnBalanceAmountEUR`
FROM dexposure as A
WHERE last_day(A.Opendate) = last_day({run_date!r}) 
AND SUBSTRING(A.ProductKey, INSTR(A.ProductKey, ';') + 1, LENGTH(A.ProductKey)) IN ('6078', '6079', '4015', '4016', '7362', '7363', '40000', '40001')
'''
)
modifications.createOrReplaceTempView("modifications")

###### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT 
  CAST(BusinessDate AS DATE) AS BusinessDate
, CAST(PortfolioName AS STRING) AS PortfolioName
, CAST(Opendate AS DATE) OpenDate
, CAST(AssetSegment AS STRING) AS AssetSegment
, CAST(CDI AS STRING) AS CDI
, CAST(PartyTaxRegistrationNumber AS STRING) AS PartyTaxRegistrationNumber
, CAST(ExposureKey AS STRING) AS ExposureKey
, CAST(AgreementKey AS STRING) AS AgreementKey
, CAST(ProductKey AS STRING) AS ProductKey
, CAST(TotalLegalClaimEUR AS DECIMAL(19,2)) AS TotalLegalClaimEUR
, CAST(GBVEUR AS DECIMAL(19,2))  AS GBVEUR
, CAST(OnBalanceAmountEUR AS DECIMAL(19,2)) AS OnBalanceAmountEUR
, CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
FROM modifications 
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
L3869Modifications = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
L3869Modifications.createOrReplaceTempView("L3869Modifications")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/HAPS/L3869Modifications/" 
file_name = "L3869Modifications_" +run_date+ ".csv"
L3869Modifications.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}") 
rename_and_cleanup_files(output_path, file_name)